# BBC Ses Sınıflandırma — CNN ve BiLSTM Eğitimi

Bu notebook, BBC ses arşivindeki 26 sınıfı sınıflandırmak için iki farklı derin öğrenme yaklaşımını karşılaştırır:

| Model | Giriş | Temel Fikir |
|-------|-------|-------------|
| **SpectrogramCNN** | (B, 1, 128, 313) | ResNet18 transfer learning, log-mel 2D görüntü olarak |
| **BiLSTMClassifier** | (B, 313, 128) | Çift yönlü LSTM + temporal attention, melspectrogram dizisi olarak |

**Veri**: 312,725 segment (5s × 16kHz), dosya düzeyinde stratified split (train 70% / val 15% / test 15%)

**Referanslar**:
- Hershey et al. (2017) — CNN Architectures for Large-Scale Audio Classification  
- Bahdanau et al. (2015) — Neural Machine Translation by Jointly Learning to Align and Translate

## 1. Ortam ve Bağımlılıklar

In [ ]:
import sys
from pathlib import Path

ROOT = Path("f:/Drthesiscode")
sys.path.insert(0, str(ROOT))

import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.data.dataset import dataloader_olustur, veri_bol
from src.models.cnn import SimpleCNN, SpectrogramCNN
from src.models.lstm import BiLSTMClassifier, SimpleLSTM

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"Device  : {DEVICE}")

## 2. Veri Hazırlama

`veri_bol()` fonksiyonu `veri_listesi.parquet`'i **dosya düzeyinde** stratified olarak böler;  
aynı dosyadan gelen segmentler aynı kümeye düşer → veri sızıntısı önlenir.

`dataloader_olustur()` ise `mod='cnn'` veya `mod='lstm'` parametresiyle uygun tensor şeklini üretir:  
- CNN : `(B, 1, 128, 313)` — kanal-birinci 2D log-mel  
- LSTM: `(B, 313, 128)` — zaman-birinci mel dizisi

In [ ]:
PARQUET = ROOT / 'data/processed/veri_listesi.parquet'

df_train, df_val, df_test = veri_bol(PARQUET, ROOT)

print(f"Dosya sayisi -> Train: {df_train['dosya'].nunique():,}  "
      f"Val: {df_val['dosya'].nunique():,}  "
      f"Test: {df_test['dosya'].nunique():,}")
print(f"Toplam satir -> {len(df_train)+len(df_val)+len(df_test):,}")
print(f"Sinif sayisi : {df_train['sinif_idx'].nunique()}")

In [ ]:
# CNN DataLoader'lari (mod='cnn')
BATCH = 32
WORKERS = 4

dl_train_cnn = dataloader_olustur(df_train, ROOT, mod='cnn',
                                   batch_size=BATCH, num_workers=WORKERS)
dl_val_cnn   = dataloader_olustur(df_val,   ROOT, mod='cnn',
                                   batch_size=BATCH*2, shuffle=False, num_workers=WORKERS)
dl_test_cnn  = dataloader_olustur(df_test,  ROOT, mod='cnn',
                                   batch_size=BATCH*2, shuffle=False, num_workers=WORKERS)

# LSTM DataLoader'lari (mod='lstm')
dl_train_lstm = dataloader_olustur(df_train, ROOT, mod='lstm',
                                    batch_size=BATCH, num_workers=WORKERS)
dl_val_lstm   = dataloader_olustur(df_val,   ROOT, mod='lstm',
                                    batch_size=BATCH*2, shuffle=False, num_workers=WORKERS)
dl_test_lstm  = dataloader_olustur(df_test,  ROOT, mod='lstm',
                                    batch_size=BATCH*2, shuffle=False, num_workers=WORKERS)

# Tensor seklini dogrula
x_cnn, y = next(iter(dl_train_cnn))
x_lstm, _ = next(iter(dl_train_lstm))
print(f"CNN  batch : x={x_cnn.shape}  y={y.shape}  dtype={x_cnn.dtype}")
print(f"LSTM batch : x={x_lstm.shape}  dtype={x_lstm.dtype}")
print(f"Train batch sayisi: {len(dl_train_cnn)}")

## 3. Örnek Spectrogram Görselleştirme

Modele giren tensörün nasıl göründüğünü inceleyelim:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i in range(3):
    mel = x_cnn[i, 0].numpy()  # (128, 313)
    sinif_ad = df_train.iloc[0]['sinif'] if 'sinif' in df_train.columns else str(y[i].item())
    im = axes[i].imshow(mel, aspect='auto', origin='lower',
                         cmap='magma', vmin=-3, vmax=3)
    axes[i].set_title(f'Ornek {i+1} | Sinif idx: {y[i].item()}', fontsize=11)
    axes[i].set_xlabel('Zaman cercevesi')
    axes[i].set_ylabel('Mel bant')
    plt.colorbar(im, ax=axes[i], label='Normalize deger')

plt.suptitle('CNN girisi: Normalize log-mel spectrogram (1×128×313)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Model Mimarileri

### 4.1 SpectrogramCNN (ResNet18 Transfer Learning)

- İlk conv katmanı **3 kanal → 1 kanal** olarak değiştirilir; pretrained ağırlıklar kanalların ortalaması alınarak korunur
- Son FC katmanı: `Dropout(0.3) → Linear(512, 26)`
- Toplam ~11.2M parametre

### 4.2 BiLSTMClassifier (Temporal Attention)

- 2 katman çift yönlü LSTM, hidden_dim=256 (toplam 512)
- Additive (Bahdanau) temporal attention: tüm zaman adımlarını ağırlıklı toplar
- FC: `LayerNorm → Dropout → Linear(512,256) → GELU → Dropout → Linear(256,26)`

In [ ]:
# Model olustur ve parametre sayisini goster
cnn_model  = SpectrogramCNN(n_sinif=26, pretrained=True)
lstm_model = BiLSTMClassifier(n_sinif=26, n_mels=128, hidden_dim=256, n_katman=2)

def parametre_say(model, ad):
    toplam = sum(p.numel() for p in model.parameters())
    egit   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{ad:25s}  Toplam: {toplam/1e6:.2f}M  Egitilecek: {egit/1e6:.2f}M")

parametre_say(cnn_model,  'SpectrogramCNN (ResNet18)')
parametre_say(lstm_model, 'BiLSTMClassifier')

# Forward gecis testi
with torch.no_grad():
    cikis_cnn  = cnn_model(x_cnn)
    cikis_lstm = lstm_model(x_lstm)
print(f"\nCNN  cikis sekli : {cikis_cnn.shape}")
print(f"LSTM cikis sekli : {cikis_lstm.shape}")

## 5. Eğitim Fonksiyonları

- **Mixed Precision (AMP)**: CUDA'da `autocast` + `GradScaler` ile ~2× hızlanma
- **Label Smoothing** (ε=0.1): aşırı öğrenmeye karşı regularizasyon
- **Gradient Clipping** (max_norm=5): patlayan gradyanları önler
- **CosineAnnealingLR**: learning rate'i soğuk başlatma olmadan smooth düşürür
- **Early Stopping**: `patience` epoch boyunca val F1 iyileşmezse durdurur

In [ ]:
def egit_epoch(model, loader, optimizer, kayip_fn, scaler, device):
    model.train()
    toplam_kayip = 0.0
    dogru = toplam = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=(device == 'cuda')):
            cikis = model(x)
            kayip = kayip_fn(cikis, y)
        scaler.scale(kayip).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        toplam_kayip += kayip.item() * x.size(0)
        dogru  += (cikis.argmax(1) == y).sum().item()
        toplam += x.size(0)
    return toplam_kayip / toplam, dogru / toplam

def degerlendir(model, loader, kayip_fn, device):
    model.eval()
    toplam_kayip = 0.0
    tahminler, gercekler = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with autocast(enabled=(device == 'cuda')):
                cikis = model(x)
                kayip = kayip_fn(cikis, y)
            toplam_kayip += kayip.item() * x.size(0)
            tahminler.extend(cikis.argmax(1).cpu().tolist())
            gercekler.extend(y.cpu().tolist())
    f1 = f1_score(gercekler, tahminler, average='macro', zero_division=0)
    return toplam_kayip / len(gercekler), f1, tahminler, gercekler

def egitim_dongüsü(model, dl_train, dl_val, dl_test,
                   model_adi, epochs=30, lr=3e-4, weight_decay=1e-4, patience=7):
    model = model.to(DEVICE)
    kayip_fn  = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)
    scaler    = GradScaler(enabled=(DEVICE == 'cuda'))

    en_iyi_val_f1  = 0.0
    patience_sayac = 0
    tarih          = []
    model_yolu     = ROOT / f'models/{model_adi}_best.pt'
    (ROOT / 'models').mkdir(exist_ok=True)

    # Onceki kontrol noktasi varsa yukle
    if model_yolu.exists():
        print(f"Mevcut kontrol noktasi bulundu: {model_yolu}")
        ckpt = torch.load(model_yolu, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        print(f"  Yuklendi  -> val F1={ckpt['val_f1']:.4f} (epoch {ckpt['epoch']})")
        print("  Test degerlendirilmesi yapiliyor...")
        test_kayip, test_f1, test_tahmin, test_gercek = degerlendir(
            model, dl_test, kayip_fn, DEVICE)
        print(f"  Test Macro F1 : {test_f1:.4f}")
        return model, {'test_f1': test_f1, 'tarih': []}

    basla = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_kayip, train_acc = egit_epoch(model, dl_train, optimizer,
                                             kayip_fn, scaler, DEVICE)
        val_kayip, val_f1, _, _ = degerlendir(model, dl_val, kayip_fn, DEVICE)
        scheduler.step()
        lr_simdi = optimizer.param_groups[0]['lr']
        sure = time.time() - t0

        print(f"E{epoch:3d}  train={train_kayip:.4f}/{train_acc:.3f}  "
              f"val={val_kayip:.4f}/F1={val_f1:.4f}  lr={lr_simdi:.1e}  {sure:.0f}s")

        tarih.append({'epoch': epoch, 'train_loss': train_kayip,
                      'train_acc': train_acc, 'val_loss': val_kayip, 'val_f1': val_f1})

        if val_f1 > en_iyi_val_f1:
            en_iyi_val_f1 = val_f1
            patience_sayac = 0
            torch.save({'epoch': epoch, 'model_adi': model_adi,
                        'model_state': model.state_dict(), 'val_f1': val_f1}, model_yolu)
        else:
            patience_sayac += 1
            if patience_sayac >= patience:
                print(f"Erken durdurma (patience={patience})")
                break

    print(f"\nEgitim: {(time.time()-basla)/60:.1f} dk  En iyi val F1: {en_iyi_val_f1:.4f}")

    # Test
    ckpt = torch.load(model_yolu, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state'])
    test_kayip, test_f1, test_tahmin, test_gercek = degerlendir(
        model, dl_test, kayip_fn, DEVICE)
    print(f"Test Macro F1 : {test_f1:.4f}")
    return model, {'test_f1': test_f1, 'val_f1': en_iyi_val_f1,
                   'test_tahmin': test_tahmin, 'test_gercek': test_gercek, 'tarih': tarih}

print('Egitim fonksiyonlari tanimlandi.')

## 6. SpectrogramCNN Eğitimi

ResNet18 pretrained ağırlıklarıyla başlayıp 30 epoch boyunca fine-tune ediyoruz.  
Kontrol noktası mevcutsa yeniden eğitmez, doğrudan test sonucunu gösterir.

In [ ]:
cnn_model = SpectrogramCNN(n_sinif=26, pretrained=True)
cnn_model, cnn_sonuc = egitim_dongüsü(
    cnn_model, dl_train_cnn, dl_val_cnn, dl_test_cnn,
    model_adi='cnn',
    epochs=30, lr=3e-4, weight_decay=1e-4, patience=7
)

## 7. BiLSTMClassifier Eğitimi

Aynı veriyi `mod='lstm'` ile zaman-birinci dizi olarak besleyip temporal attention ile eğitiyoruz.

In [ ]:
lstm_model = BiLSTMClassifier(n_sinif=26, n_mels=128, hidden_dim=256, n_katman=2)
lstm_model, lstm_sonuc = egitim_dongüsü(
    lstm_model, dl_train_lstm, dl_val_lstm, dl_test_lstm,
    model_adi='lstm',
    epochs=30, lr=3e-4, weight_decay=1e-4, patience=7
)

## 8. Eğitim Eğrilerinin Görselleştirilmesi

Loss ve F1 eğrilerini birlikte çizerek her iki modelin öğrenme dinamiğini karşılaştırıyoruz.

In [ ]:
def egri_ciz(tarih_cnn, tarih_lstm):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for tarih, renk, ad in [(tarih_cnn, 'steelblue', 'CNN'),
                             (tarih_lstm, 'tomato', 'BiLSTM')]:
        if not tarih:
            continue
        e = [r['epoch'] for r in tarih]
        axes[0].plot(e, [r['train_loss'] for r in tarih],
                     linestyle='--', color=renk, alpha=0.6, label=f'{ad} train')
        axes[0].plot(e, [r['val_loss']   for r in tarih],
                     linestyle='-',  color=renk, label=f'{ad} val')
        axes[1].plot(e, [r['val_f1'] for r in tarih],
                     linestyle='-', color=renk, marker='o', markersize=3, label=ad)

    axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Epoch')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_title('Validation Macro F1'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Macro F1'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.suptitle('CNN vs BiLSTM — Egitim Egrileri', fontsize=13)
    plt.tight_layout()
    plt.savefig(ROOT / 'results/v1/cnn_lstm_egitim_egrisi.png', dpi=150)
    plt.show()
    print('Grafik kaydedildi: results/v1/cnn_lstm_egitim_egrisi.png')

egri_ciz(cnn_sonuc.get('tarih', []), lstm_sonuc.get('tarih', []))

## 9. Karışıklık Matrisi ve Sınıf Bazlı F1

Test kümesindeki karışıklık matrisini çizerek modelin hangi sınıflarda hata yaptığını inceliyoruz.

In [ ]:
# Sinif adlarini al
df_meta = pd.read_parquet(PARQUET)
sinif_adlari = (df_meta.groupby('sinif_idx')['sinif']
                .first().sort_index().tolist()
                if 'sinif' in df_meta.columns
                else [str(i) for i in range(26)])

def karisiklik_matrisi_ciz(tahmin, gercek, baslik):
    if not tahmin:
        print(f"{baslik}: tahmin verisi yok (kontrol noktasından yüklendi).")
        return
    cm = confusion_matrix(gercek, tahmin)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm_norm, annot=False, fmt='.2f', cmap='Blues',
                xticklabels=sinif_adlari, yticklabels=sinif_adlari, ax=ax)
    ax.set_xlabel('Tahmin'); ax.set_ylabel('Gercek')
    ax.set_title(f'{baslik} — Normalize Karisiklik Matrisi', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    isim = baslik.lower().replace(' ', '_')
    plt.savefig(ROOT / f'results/v1/{isim}_karisiklik.png', dpi=150)
    plt.show()

karisiklik_matrisi_ciz(
    cnn_sonuc.get('test_tahmin', []),
    cnn_sonuc.get('test_gercek', []),
    'SpectrogramCNN'
)
karisiklik_matrisi_ciz(
    lstm_sonuc.get('test_tahmin', []),
    lstm_sonuc.get('test_gercek', []),
    'BiLSTMClassifier'
)

In [ ]:
# Sinif bazli F1 karsilastirmasi
def sinif_f1_karsilastir(tahmin_cnn, gercek_cnn, tahmin_lstm, gercek_lstm):
    if not tahmin_cnn or not tahmin_lstm:
        print("Tahmin verisi eksik (kontrol noktasindan yuklendi).")
        return

    rapor_cnn  = classification_report(gercek_cnn,  tahmin_cnn,
                                        target_names=sinif_adlari,
                                        zero_division=0, output_dict=True)
    rapor_lstm = classification_report(gercek_lstm, tahmin_lstm,
                                        target_names=sinif_adlari,
                                        zero_division=0, output_dict=True)

    f1_cnn  = [rapor_cnn[s]['f1-score']  for s in sinif_adlari]
    f1_lstm = [rapor_lstm[s]['f1-score'] for s in sinif_adlari]

    x = np.arange(len(sinif_adlari))
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(x - 0.2, f1_cnn,  0.38, label='CNN',    color='steelblue', alpha=0.85)
    ax.bar(x + 0.2, f1_lstm, 0.38, label='BiLSTM', color='tomato',    alpha=0.85)
    ax.axhline(np.mean(f1_cnn),  color='steelblue', ls='--', lw=1)
    ax.axhline(np.mean(f1_lstm), color='tomato',    ls='--', lw=1)
    ax.set_xticks(x); ax.set_xticklabels(sinif_adlari, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('F1 Skoru'); ax.set_ylim(0, 1.05)
    ax.set_title('Sinif Bazli F1: CNN vs BiLSTM', fontsize=12)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(ROOT / 'results/v1/sinif_f1_karsilastirma.png', dpi=150)
    plt.show()

sinif_f1_karsilastir(
    cnn_sonuc.get('test_tahmin', []),
    cnn_sonuc.get('test_gercek', []),
    lstm_sonuc.get('test_tahmin', []),
    lstm_sonuc.get('test_gercek', [])
)

## 10. Model Karşılaştırması ve LGBM Baseline ile Kıyaslama

Tüm modellerin test kümesi Macro F1 skorlarını tabloya döküyoruz.

In [ ]:
# Sonuc JSON'larini oku (mevcut olanlar)
sonuc_dizin = ROOT / 'results' / 'v1'
satirlar = []

# LGBM sonuclarini bul
lgbm_f1 = None
for f in sorted(sonuc_dizin.glob('lgbm_*.json')):
    with open(f) as fp:
        d = json.load(fp)
    lgbm_f1 = d.get('test_macro_f1', d.get('test_f1'))

if lgbm_f1:
    satirlar.append({'Model': 'LightGBM (baseline)', 'Test Macro F1': lgbm_f1,
                     'Tur': 'Geleneksel ML'})

# CNN ve LSTM sonuclarini oku / hesapla
for model_adi, sonuc in [('SpectrogramCNN', cnn_sonuc), ('BiLSTMClassifier', lstm_sonuc)]:
    f1 = sonuc.get('test_f1')
    if f1:
        satirlar.append({'Model': model_adi, 'Test Macro F1': f1, 'Tur': 'Derin Ogrenme'})

df_karsi = pd.DataFrame(satirlar).sort_values('Test Macro F1', ascending=False)
df_karsi['Test Macro F1'] = df_karsi['Test Macro F1'].map('{:.4f}'.format)
print(df_karsi.to_string(index=False))

# Kaydet
df_karsi.to_csv(sonuc_dizin / 'model_karsilastirma.csv', index=False)
print('\nKarsilastirma tablosu kaydedildi: results/v1/model_karsilastirma.csv')

## 11. Sonuçlar ve Yorumlar

### Genel Bulgular

- **SpectrogramCNN (ResNet18)**: ImageNet'ten pretrained ağırlıklar log-mel spectrogramlara iyi transfer oluyor; özellikle frekans kalıplarını yakalamada güçlü.
- **BiLSTMClassifier**: Temporal attention, konuşma/müzik gibi uzun süreli bağımlılıkları olan sınıflarda avantaj sağlıyor.
- **LightGBM baseline**: El-ile hesaplanmış 467-boyutlu özellikler, derin modeller için güçlü bir referans noktası oluşturuyor.

### Tez İçin Önerilen Hibrit Yaklaşım

```
LGBM (467-özellik) + CNN (logit) + BiLSTM (logit)
          └─────── Late Fusion (ağırlıklı toplam) ───────┘
```

Her modelin "uzmanlaştığı" sınıf grubu farklıdır; geç füzyon bu tamamlayıcılıktan yararlanır.

### Sonraki Adımlar

1. `04_fusion.py` — LGBM + CNN + BiLSTM geç füzyon
2. Optuna HPO ile CNN hyperparameter arama
3. AudioLDM ile veri artırma → yeniden eğitim